# Backtracking, Pruning, and Search States


```{contents}
:local:
:depth: 2
```


```{index} Undo Exactly the Choice Owned by the Current Call
```



```{index} Undo Exactly the Choice Owned by the Current Call
```

## Undo Exactly the Choice Owned by the Current Call



Backtracking explores alternatives while preserving enough state to return to a previous decision point. For one-use invoice reconciliation, each position can be included or excluded. A helper at position i with remaining target r represents a particular smaller problem: can some subset of positions i onward total r? Both i and r matter because they describe which choices remain.

For amounts 2, 3 and 5 with target five, one witness is 2 plus 3 and another is 5 alone. Choosing the first invoice changes the remaining target from five to three and advances the position. Excluding it leaves target five but still advances the position. If both branches kept the same position, the program could reuse an invoice whose contract permits only one use.

| Branch from position 0 | Next position | Remaining target |
| --- | --- | --- |
| Include amount 2 | 1 | 3 |
| Exclude amount 2 | 1 | 5 |

When building a selected-position list, add the position for the include branch, explore, and remove that same choice when returning to explore an alternative. A saved result must be a snapshot if later mutations would otherwise change it. Assigning a result variable to the mutable working list only creates another reference to the changing list.

Signed amounts require care with pruning. The common shortcut remaining target below zero means impossible is unsafe when later negative adjustments can reduce a sum. Bounds based on the sums of remaining negative and positive amounts provide a justified possible range. Prune only when the requested remaining value lies outside that range. Pruning changes how much is explored; it must not discard a feasible solution.

Caching failed states can avoid revisiting the same impossible suffix/target pair. The cache key must identify the complete state under the fixed input and policy. A call budget is a separate implementation limit: exhausting it means the search did not finish, not that infeasibility has been proved. Preserve this distinction when reporting results to a user.

Guided trace: add a -2 adjustment after a temporarily excessive positive choice. Construct a target for which pruning on a negative remainder would lose a valid witness. Then compare a proven failed state with a stopped-by-budget state. Explain why only the former may safely be reused as evidence of impossibility.

For related explanation, see the further reading below.[^development-reading]


```{index} Backtracking and Search Spaces
```

## Backtracking and Search Spaces



Backtracking is depth-first search through decisions. An invariant describes the partial choice; a base case records a complete result; undo restores shared state before a sibling branch. Pruning must exclude only impossible branches. Returning a witness allows the caller to check the decision.

```{index} Choose, Explore, Undo, and Snapshot
```

## Choose, Explore, Undo, and Snapshot




```{index} Choose, Explore, Undo, and Snapshot
```

At position i, chosen holds exactly the included positions from 0..i−1. Include i, recurse to i+1, undo, then exclude i. At i=n, snapshot chosen. Positions distinguish equal-valued business records; choosing position zero twice is not allowed.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

var subsets=ChoicePlanning.Subsets(3);
Console.WriteLine(string.Join(" | ",subsets.Select(s=>"["+string.Join(",",s)+"]")));
Console.WriteLine($"count={subsets.Length}, empty-input results={ChoicePlanning.Subsets(0).Length}");
subsets[0][0]=99;
Console.WriteLine("independent next snapshot="+string.Join(",",subsets[1]));
public static class ChoicePlanning
{
    public static int[][] Subsets(int n)
    {
        if(n<0||n>16)throw new ArgumentOutOfRangeException(nameof(n));
        var result=new List<int[]>();var chosen=new List<int>();
        void Visit(int index)
        {
            if(index==n){result.Add(chosen.ToArray());return;}
            chosen.Add(index);Visit(index+1);chosen.RemoveAt(chosen.Count-1);
            Visit(index+1);
        }
        Visit(0);return result.ToArray();
    }
    public static int[][] Permutations(int n)
    {
        if(n<0||n>8)throw new ArgumentOutOfRangeException(nameof(n));
        var result=new List<int[]>();var chosen=new List<int>();var used=new bool[n];
        void Visit()
        {
            if(chosen.Count==n){result.Add(chosen.ToArray());return;}
            for(int i=0;i<n;i++)
            {
                if(used[i])continue;
                used[i]=true;chosen.Add(i);Visit();chosen.RemoveAt(chosen.Count-1);used[i]=false;
            }
        }
        Visit();return result.ToArray();
    }
}


[0,1,2] | [0,1] | [0,2] | [0] | [1,2] | [1] | [2] | []
count=8, empty-input results=1
independent next snapshot=0,1


The include/exclude branches are disjoint and exhaustive, and each step reduces unprocessed positions, proving termination and completeness. Copying at a leaf prevents later undo from erasing saved results. There are 2^n leaves and 2^(n+1)−1 recursive calls. Total copied position entries are n·2^(n−1) for n≥1: enumeration takes Θ(n2^n) time and retained output space, even though the active stack and chosen list use only O(n). The teaching API limits n to 16 to bound retained output. Empty input has one subset, the empty subset.

**Common mistake.** Adding the shared `chosen` list itself to the results instead of a copy. Every later undo edits the stored subsets too, and the final list holds `n` references to one empty list; the snapshot at the leaf is what the `subsets[0][0] = 99` line checks.


```{index} One-Use Subset Sum Needs a Two-Part State
```

## One-Use Subset Sum Needs a Two-Part State




```{index} One-Use Subset Sum Needs a Two-Part State
```

For signed invoice adjustments, ask whether some distinct positions sum to target. State (i,r) means: using only positions i..n−1, can we make remaining r? Include gives (i+1,r−value[i]); exclude gives (i+1,r). A zero remainder accepts the current subset, including the empty subset for target zero.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

int[] values={8,-3,4};
var result=SubsetPlanning.FindSubset(values,5);
Console.WriteLine($"indices=[{string.Join(",",result.Indices!)}], values=[{string.Join(",",result.Indices!.Select(i=>values[i]))}]");
Console.WriteLine($"sum={result.Indices!.Sum(i=>(long)values[i])}, calls={result.Calls}");
Console.WriteLine($"empty target witness length={SubsetPlanning.FindSubset(Array.Empty<int>(),0).Indices!.Length}");
public static class SubsetPlanning
{
    public record SubsetResult(int[]? Indices,int Calls,int Hits,int Prunes,int FailedStates);
    // Signed values are allowed; positions are distinct and each is used once.
    public static SubsetResult FindSubset(int[] values,long target,bool prune=true,bool memoize=true,int maxCalls=1_000_000)
    {
        ArgumentNullException.ThrowIfNull(values);
        if(values.Length>128)throw new ArgumentException("At most 128 positions in this teaching implementation.");
        if(maxCalls<1)throw new ArgumentOutOfRangeException(nameof(maxCalls));
        int[] data=(int[])values.Clone();int n=data.Length;
        var lower=new long[n+1];var upper=new long[n+1];
        for(int i=n-1;i>=0;i--)
        {lower[i]=lower[i+1]+Math.Min(0,data[i]);upper[i]=upper[i+1]+Math.Max(0,data[i]);}
        var failed=new HashSet<(int Index,long Remaining)>();var chosen=new List<int>();
        int calls=0,hits=0,prunes=0;int[]? answer=null;
        bool Search(int index,long remaining)
        {
            if(calls==maxCalls)throw new InvalidOperationException("Search budget exhausted; feasibility is unknown.");
            calls++;
            if(remaining==0){answer=chosen.ToArray();return true;}
            if(index==n)return false;
            if(prune&&(remaining<lower[index]||remaining>upper[index])){prunes++;return false;}
            var state=(index,remaining);
            if(memoize&&failed.Contains(state)){hits++;return false;}
            chosen.Add(index);bool found;
            try{found=Search(index+1,remaining-data[index]);}
            finally{chosen.RemoveAt(chosen.Count-1);}
            if(found||Search(index+1,remaining))return true;
            if(memoize)failed.Add(state);
            return false;
        }
        // Even an unpruned trace rejects targets outside the total signed range.
        if(target<lower[0]||target>upper[0]){calls=1;prunes=1;}
        else Search(0,target);
        if(chosen.Count!=0)throw new InvalidOperationException("Undo failed.");
        return new SubsetResult(answer,calls,hits,prunes,failed.Count);
    }
}


indices=[0,1], values=[8,-3]
sum=5, calls=3
empty target witness length=0


Each call advances i, so no position is reused and recursion terminates. The shared chosen list contains selected earlier positions; remaining equals target minus their sum. The implementation snapshots a successful witness and restores the list even on early success or a search-budget exception. A null witness means proven infeasibility; an empty witness is a feasible zero target. Use long sums and remaining amounts so two int values do not overflow.

**Common mistake.** Recursing on `(i, r - value)` without advancing `i`. The same position can then be chosen again, turning the one-use invoices into unlimited coins, and the search may report a witness that uses one record twice.


```{index} Prove Bounds Before Pruning
```

## Prove Bounds Before Pruning




```{index} Prove Bounds Before Pruning
```

The rule remaining<0 is safe for nonnegative values because future additions cannot decrease the chosen sum. It fails for signed values: 8 followed by −3 makes five after temporarily overshooting. For a signed suffix define L(i) as the sum of its negative values and U(i) as the sum of its positive values. Every suffix subset sum lies in [L(i),U(i)]. Prune only if r lies outside those bounds. Being inside the interval does not guarantee feasibility.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

int[] values={2,4,6,9};
var raw=SubsetPlanning.FindSubset(values,8,false,false);
var pruned=SubsetPlanning.FindSubset(values,8,true,false);
Console.WriteLine($"raw calls={raw.Calls}; pruned calls={pruned.Calls}, prunes={pruned.Prunes}");
var gap=SubsetPlanning.FindSubset(new[]{4,6},5);
Console.WriteLine($"inside bounds but feasible={gap.Indices is not null}");
var signed=SubsetPlanning.FindSubset(new[]{8,-3},5);
Console.WriteLine($"signed witness={string.Join(",",signed.Indices!)}");
public static class SubsetPlanning
{
    public record SubsetResult(int[]? Indices,int Calls,int Hits,int Prunes,int FailedStates);
    // Signed values are allowed; positions are distinct and each is used once.
    public static SubsetResult FindSubset(int[] values,long target,bool prune=true,bool memoize=true,int maxCalls=1_000_000)
    {
        ArgumentNullException.ThrowIfNull(values);
        if(values.Length>128)throw new ArgumentException("At most 128 positions in this teaching implementation.");
        if(maxCalls<1)throw new ArgumentOutOfRangeException(nameof(maxCalls));
        int[] data=(int[])values.Clone();int n=data.Length;
        var lower=new long[n+1];var upper=new long[n+1];
        for(int i=n-1;i>=0;i--)
        {lower[i]=lower[i+1]+Math.Min(0,data[i]);upper[i]=upper[i+1]+Math.Max(0,data[i]);}
        var failed=new HashSet<(int Index,long Remaining)>();var chosen=new List<int>();
        int calls=0,hits=0,prunes=0;int[]? answer=null;
        bool Search(int index,long remaining)
        {
            if(calls==maxCalls)throw new InvalidOperationException("Search budget exhausted; feasibility is unknown.");
            calls++;
            if(remaining==0){answer=chosen.ToArray();return true;}
            if(index==n)return false;
            if(prune&&(remaining<lower[index]||remaining>upper[index])){prunes++;return false;}
            var state=(index,remaining);
            if(memoize&&failed.Contains(state)){hits++;return false;}
            chosen.Add(index);bool found;
            try{found=Search(index+1,remaining-data[index]);}
            finally{chosen.RemoveAt(chosen.Count-1);}
            if(found||Search(index+1,remaining))return true;
            if(memoize)failed.Add(state);
            return false;
        }
        // Even an unpruned trace rejects targets outside the total signed range.
        if(target<lower[0]||target>upper[0]){calls=1;prunes=1;}
        else Search(0,target);
        if(chosen.Count!=0)throw new InvalidOperationException("Undo failed.");
        return new SubsetResult(answer,calls,hits,prunes,failed.Count);
    }
}


raw calls=11; pruned calls=9, prunes=1
inside bounds but feasible=False
signed witness=0,1


Compute suffix bounds once in O(n) time/storage. The code checks success before suffix exhaustion, preserves both branches when necessary, and accepts zeros. The initial total bound check also runs in the unpruned comparison to avoid subtracting from an extreme impossible long target. Branch order affects measured calls when the search stops at its first witness; pruning has no universal speedup guarantee.

**Common mistake.** Pruning on `remaining < 0` with signed values. The `8, -3` witness for target 5 overshoots to 8 first and is cut off before the `-3` can bring it back; the suffix bounds are the version of the test that is safe for signed inputs.


```{index} Memoize Failed States Without Losing Choices
```

## Memoize Failed States Without Losing Choices




```{index} Memoize Failed States Without Losing Choices
```

Two different prefixes can leave the same (i,r). Their future feasibility is identical because the remaining positions and target are identical. Cache a state only after both branches fail. Do not cache only r: the same remaining amount with different inventories is a different question. Keeping only failed states avoids needing to store a prefix-dependent witness.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

int[] values=Enumerable.Repeat(2,12).ToArray();
var raw=SubsetPlanning.FindSubset(values,11,false,false);
var pruned=SubsetPlanning.FindSubset(values,11,true,false);
var memo=SubsetPlanning.FindSubset(values,11,true,true);
Console.WriteLine($"raw={raw.Calls}, bounds={pruned.Calls}, memo={memo.Calls}, hits={memo.Hits}, failed states={memo.FailedStates}");
Console.WriteLine($"feasible={memo.Indices is not null}");
public static class SubsetPlanning
{
    public record SubsetResult(int[]? Indices,int Calls,int Hits,int Prunes,int FailedStates);
    // Signed values are allowed; positions are distinct and each is used once.
    public static SubsetResult FindSubset(int[] values,long target,bool prune=true,bool memoize=true,int maxCalls=1_000_000)
    {
        ArgumentNullException.ThrowIfNull(values);
        if(values.Length>128)throw new ArgumentException("At most 128 positions in this teaching implementation.");
        if(maxCalls<1)throw new ArgumentOutOfRangeException(nameof(maxCalls));
        int[] data=(int[])values.Clone();int n=data.Length;
        var lower=new long[n+1];var upper=new long[n+1];
        for(int i=n-1;i>=0;i--)
        {lower[i]=lower[i+1]+Math.Min(0,data[i]);upper[i]=upper[i+1]+Math.Max(0,data[i]);}
        var failed=new HashSet<(int Index,long Remaining)>();var chosen=new List<int>();
        int calls=0,hits=0,prunes=0;int[]? answer=null;
        bool Search(int index,long remaining)
        {
            if(calls==maxCalls)throw new InvalidOperationException("Search budget exhausted; feasibility is unknown.");
            calls++;
            if(remaining==0){answer=chosen.ToArray();return true;}
            if(index==n)return false;
            if(prune&&(remaining<lower[index]||remaining>upper[index])){prunes++;return false;}
            var state=(index,remaining);
            if(memoize&&failed.Contains(state)){hits++;return false;}
            chosen.Add(index);bool found;
            try{found=Search(index+1,remaining-data[index]);}
            finally{chosen.RemoveAt(chosen.Count-1);}
            if(found||Search(index+1,remaining))return true;
            if(memoize)failed.Add(state);
            return false;
        }
        // Even an unpruned trace rejects targets outside the total signed range.
        if(target<lower[0]||target>upper[0]){calls=1;prunes=1;}
        else Search(0,target);
        if(chosen.Count!=0)throw new InvalidOperationException("Undo failed.");
        return new SubsetResult(answer,calls,hits,prunes,failed.Count);
    }
}


raw=8191, bounds=3431, memo=85, hits=30, failed states=42
feasible=False


For nonnegative values and target T, reachable non-pruned remainder states lie in 0..T, so there are O(n(T+1)) possible states. Signed inputs instead depend on the attainable sum range, which can be large. Expected constant-time hash lookup gives work proportional to visited states/transitions plus pruning and terminal calls. Without effective caching or pruning, worst-case branching is O(2^n), with O(n) active stack/path space. This teaching implementation accepts up to 128 positions and a default million-call budget. Exhausting it throws with feasibility unknown; it must never return false as though the instance were solved.

**Common mistake.** Caching `r` alone as failed. The same remaining amount reached with different positions left is a different question, and a later state with more inventory would be wrongly declared infeasible.


```{index} Permutations Use Every Position Exactly Once
```

## Permutations Use Every Position Exactly Once




```{index} Permutations Use Every Position Exactly Once
```

At each depth choose an unused position. Maintain used[i] exactly when i appears in order, and undo both structures. Labels can repeat while positions remain distinct. This is useful when two jobs share a display name but have different IDs.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

string[] labels={"Audit","Audit","Review"};
var orders=ChoicePlanning.Permutations(labels.Length);
foreach(var order in orders)Console.WriteLine($"IDs={string.Join(",",order)}; labels={string.Join("/",order.Select(i=>labels[i]))}");
Console.WriteLine($"position orders={orders.Length}, distinct label strings={orders.Select(o=>string.Join("/",o.Select(i=>labels[i]))).Distinct().Count()}");
Console.WriteLine($"empty-input orders={ChoicePlanning.Permutations(0).Length}");
public static class ChoicePlanning
{
    public static int[][] Subsets(int n)
    {
        if(n<0||n>16)throw new ArgumentOutOfRangeException(nameof(n));
        var result=new List<int[]>();var chosen=new List<int>();
        void Visit(int index)
        {
            if(index==n){result.Add(chosen.ToArray());return;}
            chosen.Add(index);Visit(index+1);chosen.RemoveAt(chosen.Count-1);
            Visit(index+1);
        }
        Visit(0);return result.ToArray();
    }
    public static int[][] Permutations(int n)
    {
        if(n<0||n>8)throw new ArgumentOutOfRangeException(nameof(n));
        var result=new List<int[]>();var chosen=new List<int>();var used=new bool[n];
        void Visit()
        {
            if(chosen.Count==n){result.Add(chosen.ToArray());return;}
            for(int i=0;i<n;i++)
            {
                if(used[i])continue;
                used[i]=true;chosen.Add(i);Visit();chosen.RemoveAt(chosen.Count-1);used[i]=false;
            }
        }
        Visit();return result.ToArray();
    }
}


IDs=0,1,2; labels=Audit/Audit/Review
IDs=0,2,1; labels=Audit/Review/Audit
IDs=1,0,2; labels=Audit/Audit/Review
IDs=1,2,0; labels=Audit/Review/Audit
IDs=2,0,1; labels=Review/Audit/Audit
IDs=2,1,0; labels=Review/Audit/Audit
position orders=6, distinct label strings=3
empty-input orders=1


There are n! position permutations; copying each length-n order requires Θ(n·n!) time and retained output space. Active state is O(n). The API caps n at eight for materialized results. Generating unique value permutations is a different contract and would need deliberate duplicate suppression. A streamed generator can reduce retained storage, but cannot eliminate the cost of producing all requested outputs.

**Common mistake.** Forgetting to reset `used[i]` after the recursive call returns. The second branch then sees position `i` as taken, and the generator produces fewer than `n!` orders without any error.


```{index} Exercise
```

## Exercise




```{index} Exercise
```

Find a signed, wide-sum witness using each position once, reject an impossible target, and distinguish a budget limit from infeasibility.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

int[] values={int.MaxValue,int.MaxValue,-1};
// Your code starts here.
/* TODO: complete the task. */
// Your code ends here.

In [1]:
using System;
using System.Linq;
using System.Collections.Generic;

// Solution
int[] values={int.MaxValue,int.MaxValue,-1};
var found=SubsetPlanning.FindSubset(values,2L*int.MaxValue-1);
Console.WriteLine($"sum={found.Indices!.Sum(i=>(long)values[i])}, IDs={string.Join(",",found.Indices!)}");
Console.WriteLine($"impossible={SubsetPlanning.FindSubset(values,long.MinValue).Indices is null}");
try{SubsetPlanning.FindSubset(new[]{2,2,2},3,false,false,1);}catch(InvalidOperationException){Console.WriteLine("budget exhausted: feasibility unknown");}
public static class SubsetPlanning
{
    public record SubsetResult(int[]? Indices,int Calls,int Hits,int Prunes,int FailedStates);
    // Signed values are allowed; positions are distinct and each is used once.
    public static SubsetResult FindSubset(int[] values,long target,bool prune=true,bool memoize=true,int maxCalls=1_000_000)
    {
        ArgumentNullException.ThrowIfNull(values);
        if(values.Length>128)throw new ArgumentException("At most 128 positions in this teaching implementation.");
        if(maxCalls<1)throw new ArgumentOutOfRangeException(nameof(maxCalls));
        int[] data=(int[])values.Clone();int n=data.Length;
        var lower=new long[n+1];var upper=new long[n+1];
        for(int i=n-1;i>=0;i--)
        {lower[i]=lower[i+1]+Math.Min(0,data[i]);upper[i]=upper[i+1]+Math.Max(0,data[i]);}
        var failed=new HashSet<(int Index,long Remaining)>();var chosen=new List<int>();
        int calls=0,hits=0,prunes=0;int[]? answer=null;
        bool Search(int index,long remaining)
        {
            if(calls==maxCalls)throw new InvalidOperationException("Search budget exhausted; feasibility is unknown.");
            calls++;
            if(remaining==0){answer=chosen.ToArray();return true;}
            if(index==n)return false;
            if(prune&&(remaining<lower[index]||remaining>upper[index])){prunes++;return false;}
            var state=(index,remaining);
            if(memoize&&failed.Contains(state)){hits++;return false;}
            chosen.Add(index);bool found;
            try{found=Search(index+1,remaining-data[index]);}
            finally{chosen.RemoveAt(chosen.Count-1);}
            if(found||Search(index+1,remaining))return true;
            if(memoize)failed.Add(state);
            return false;
        }
        // Even an unpruned trace rejects targets outside the total signed range.
        if(target<lower[0]||target>upper[0]){calls=1;prunes=1;}
        else Search(0,target);
        if(chosen.Count!=0)throw new InvalidOperationException("Undo failed.");
        return new SubsetResult(answer,calls,hits,prunes,failed.Count);
    }
}


sum=4294967293, IDs=0,1,2
impossible=True
budget exhausted: feasibility unknown


The output is `sum=4294967293, IDs=0,1,2`, `impossible=True`, and `budget exhausted: feasibility unknown`. The witness uses all three positions and its sum is `2 * int.MaxValue - 1`, which only fits because sums are accumulated in `long`. The third line is not a failure: a budget limit means the search stopped before deciding, which is a different outcome from a proven `null`.


```{index} Check Search Against Exhaustive Positions
```

## Check Search Against Exhaustive Positions




```{index} Check Search Against Exhaustive Positions
```

Enumerate all subsets of four signed invoice positions. Compare existence with the pruned/memoized search and validate its returned indices and sum. Do not reuse its pruning rule in the reference.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

int[] invoices={8,-3,4,4};int target=5;
bool expected=ChoicePlanning.Subsets(invoices.Length).Any(ids=>ids.Sum(i=>(long)invoices[i])==target);
var result=SubsetPlanning.FindSubset(invoices,target);
bool valid=result.Indices is not null&&result.Indices.Distinct().Count()==result.Indices.Length&&result.Indices.All(i=>i>=0&&i<invoices.Length)&&result.Indices.Sum(i=>(long)invoices[i])==target;
if(expected!=(result.Indices is not null)||!valid)throw new Exception("Subset reference disagreement.");
Console.WriteLine($"exhaustive feasible={expected}, witness valid={valid}, IDs={string.Join(",",result.Indices!)}");
public static class SubsetPlanning
{
    public record SubsetResult(int[]? Indices,int Calls,int Hits,int Prunes,int FailedStates);
    // Signed values are allowed; positions are distinct and each is used once.
    public static SubsetResult FindSubset(int[] values,long target,bool prune=true,bool memoize=true,int maxCalls=1_000_000)
    {
        ArgumentNullException.ThrowIfNull(values);
        if(values.Length>128)throw new ArgumentException("At most 128 positions in this teaching implementation.");
        if(maxCalls<1)throw new ArgumentOutOfRangeException(nameof(maxCalls));
        int[] data=(int[])values.Clone();int n=data.Length;
        var lower=new long[n+1];var upper=new long[n+1];
        for(int i=n-1;i>=0;i--)
        {lower[i]=lower[i+1]+Math.Min(0,data[i]);upper[i]=upper[i+1]+Math.Max(0,data[i]);}
        var failed=new HashSet<(int Index,long Remaining)>();var chosen=new List<int>();
        int calls=0,hits=0,prunes=0;int[]? answer=null;
        bool Search(int index,long remaining)
        {
            if(calls==maxCalls)throw new InvalidOperationException("Search budget exhausted; feasibility is unknown.");
            calls++;
            if(remaining==0){answer=chosen.ToArray();return true;}
            if(index==n)return false;
            if(prune&&(remaining<lower[index]||remaining>upper[index])){prunes++;return false;}
            var state=(index,remaining);
            if(memoize&&failed.Contains(state)){hits++;return false;}
            chosen.Add(index);bool found;
            try{found=Search(index+1,remaining-data[index]);}
            finally{chosen.RemoveAt(chosen.Count-1);}
            if(found||Search(index+1,remaining))return true;
            if(memoize)failed.Add(state);
            return false;
        }
        // Even an unpruned trace rejects targets outside the total signed range.
        if(target<lower[0]||target>upper[0]){calls=1;prunes=1;}
        else Search(0,target);
        if(chosen.Count!=0)throw new InvalidOperationException("Undo failed.");
        return new SubsetResult(answer,calls,hits,prunes,failed.Count);
    }
}

public static class ChoicePlanning
{
    public static int[][] Subsets(int n)
    {
        if(n<0||n>16)throw new ArgumentOutOfRangeException(nameof(n));
        var result=new List<int[]>();var chosen=new List<int>();
        void Visit(int index)
        {
            if(index==n){result.Add(chosen.ToArray());return;}
            chosen.Add(index);Visit(index+1);chosen.RemoveAt(chosen.Count-1);
            Visit(index+1);
        }
        Visit(0);return result.ToArray();
    }
    public static int[][] Permutations(int n)
    {
        if(n<0||n>8)throw new ArgumentOutOfRangeException(nameof(n));
        var result=new List<int[]>();var chosen=new List<int>();var used=new bool[n];
        void Visit()
        {
            if(chosen.Count==n){result.Add(chosen.ToArray());return;}
            for(int i=0;i<n;i++)
            {
                if(used[i])continue;
                used[i]=true;chosen.Add(i);Visit();chosen.RemoveAt(chosen.Count-1);used[i]=false;
            }
        }
        Visit();return result.ToArray();
    }
}


exhaustive feasible=True, witness valid=True, IDs=0,1


Exhaustive verification is useful on tiny instances; it is not a production solution for large inputs. Check zero targets and empty input separately because an empty witness is a successful result, not a missing one.

**Common mistake.** Reusing the search's own pruning rule inside the reference. A reference that shares the implementation's assumption cannot catch a wrong assumption; the exhaustive check enumerates subsets blindly on purpose.


```{index} Explain a Recommendation with Evidence
```

## Explain a Recommendation with Evidence




```{index} Explain a Recommendation with Evidence
```

Choose a method from objective, inventory, state range, requested witness, and output volume. Unlimited minimum-pack counts suggest amount DP with predecessor recovery. One-use signed records need a position-aware state; backtracking can stop at the first witness and safely prune signed bounds. Dense manageable state ranges may favor tabulation; sparse visited states may favor memoization.

Report measured calls for a fixed instance alongside worst-case limits. Memoization stores visited states; rolling Fibonacci discards history; materialized permutations retain factorial output. Avoid selecting an algorithm from a keyword in a scenario string. Describe which dependency, constraint, or required result makes the method suitable.

```{index} Exercise
```

## Practice: Validate Search Evidence




```{index} Exercise
```

Check empty input, zero target, unreachable quantity, and independent witness snapshots. Explain which result means success.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

int[] sizes=Array.Empty<int>();
// Your code starts here.
/* TODO: complete the task. */
// Your code ends here.

In [1]:
using System;
using System.Linq;
using System.Collections.Generic;

// Solution
int[] sizes=Array.Empty<int>();
var zero=CoinPlanning.MinimumCoins(sizes,0);var missing=CoinPlanning.MinimumCoins(sizes,1);
Console.WriteLine($"zero count={zero.Count}, missing reachable={missing.Count is not null}");
var a=SubsetPlanning.FindSubset(new[]{3,3},6);var b=SubsetPlanning.FindSubset(new[]{3,3},6);a.Indices![0]=99;
Console.WriteLine($"independent witness={string.Join(",",b.Indices!)}");
Console.WriteLine($"empty subset feasible={SubsetPlanning.FindSubset(Array.Empty<int>(),0).Indices is not null}");
public static class CoinPlanning
{
    public record CoinPlan(int? Count,int[] Coins,int?[] Best);
    // Unlimited positive denominations. A target cap bounds table allocation.
    public static CoinPlan MinimumCoins(int[] denominations,int target)
    {
        ArgumentNullException.ThrowIfNull(denominations);
        if(target<0||target>1_000_000)throw new ArgumentOutOfRangeException(nameof(target));
        if(denominations.Any(c=>c<=0))throw new ArgumentException("Coins must be positive.");
        int[] coins=denominations.Distinct().OrderBy(c=>c).ToArray();
        var best=new int?[target+1];var last=new int[target+1];best[0]=0;
        for(int amount=1;amount<=target;amount++)
        {
            foreach(int coin in coins)
            {
                if(coin>amount)break;
                if(best[amount-coin] is not int count)continue;
                int candidate=count+1;
                if(best[amount] is null||candidate<best[amount])
                {best[amount]=candidate;last[amount]=coin;}
            }
        }
        var chosen=new List<int>();
        if(best[target] is not null)
            for(int remaining=target;remaining>0;remaining-=last[remaining])chosen.Add(last[remaining]);
        return new CoinPlan(best[target],chosen.ToArray(),best);
    }
}

public static class SubsetPlanning
{
    public record SubsetResult(int[]? Indices,int Calls,int Hits,int Prunes,int FailedStates);
    // Signed values are allowed; positions are distinct and each is used once.
    public static SubsetResult FindSubset(int[] values,long target,bool prune=true,bool memoize=true,int maxCalls=1_000_000)
    {
        ArgumentNullException.ThrowIfNull(values);
        if(values.Length>128)throw new ArgumentException("At most 128 positions in this teaching implementation.");
        if(maxCalls<1)throw new ArgumentOutOfRangeException(nameof(maxCalls));
        int[] data=(int[])values.Clone();int n=data.Length;
        var lower=new long[n+1];var upper=new long[n+1];
        for(int i=n-1;i>=0;i--)
        {lower[i]=lower[i+1]+Math.Min(0,data[i]);upper[i]=upper[i+1]+Math.Max(0,data[i]);}
        var failed=new HashSet<(int Index,long Remaining)>();var chosen=new List<int>();
        int calls=0,hits=0,prunes=0;int[]? answer=null;
        bool Search(int index,long remaining)
        {
            if(calls==maxCalls)throw new InvalidOperationException("Search budget exhausted; feasibility is unknown.");
            calls++;
            if(remaining==0){answer=chosen.ToArray();return true;}
            if(index==n)return false;
            if(prune&&(remaining<lower[index]||remaining>upper[index])){prunes++;return false;}
            var state=(index,remaining);
            if(memoize&&failed.Contains(state)){hits++;return false;}
            chosen.Add(index);bool found;
            try{found=Search(index+1,remaining-data[index]);}
            finally{chosen.RemoveAt(chosen.Count-1);}
            if(found||Search(index+1,remaining))return true;
            if(memoize)failed.Add(state);
            return false;
        }
        // Even an unpruned trace rejects targets outside the total signed range.
        if(target<lower[0]||target>upper[0]){calls=1;prunes=1;}
        else Search(0,target);
        if(chosen.Count!=0)throw new InvalidOperationException("Undo failed.");
        return new SubsetResult(answer,calls,hits,prunes,failed.Count);
    }
}


zero count=0, missing reachable=False
independent witness=0,1
empty subset feasible=True


The output is `zero count=0, missing reachable=False`, `independent witness=0,1`, and `empty subset feasible=True`. With no denominations, target 0 needs zero coins and target 1 is unreachable; the second search's witness was untouched by the edit to the first, because each result owns its array; and the empty subset is a real answer for target 0.


```{rubric} Footnotes
```


[^2502_backtracking-1]: A bound is necessary for feasibility, not sufficient. Returning true merely because a target is between bounds is incorrect.
[^2502_backtracking-2]: Complexity includes output copying. Retaining every subset or permutation can dominate the active recursion space.

[^2503_optimization_lab-1]: Match the reference to the exact objective and inventory policy; a reusable-coin oracle does not verify one-use subset search.
[^2503_optimization_lab-2]: A search budget is a resource policy. Its exhaustion reports an unknown answer, distinct from proven infeasibility.

[^development-reading]: Further reading: [OpenDSA, Dynamic Programming](https://opendsa.org/OpenDSA/Books/CS4104/html/DynamicProgramming.html). The worked business scenarios and guided reasoning here are original teaching examples.
